# Project 61 — Local Prompt Evaluation Lab

**Systematically compare prompt variants across multiple dimensions — all locally.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Analysis | pandas |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to design a **prompt evaluation framework** with test cases and scoring criteria
2. How to run **controlled experiments** comparing prompt variants
3. How to use **LLM-as-judge** for automated quality scoring
4. How to analyze results with **statistical summaries** and comparisons
5. How to iterate on prompts using **data-driven insights**

## 2 · Why This Project Matters

Prompt engineering is often done by trial and error. A systematic evaluation lab
lets you compare prompts objectively, track improvements over time, and make
confident decisions about which prompt to ship. Running locally means you can
iterate fast without API costs.

## 3 · Environment Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports and Configuration

In [ ]:
import json
import time
import pandas as pd

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.3)
judge_llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Define Test Cases

Good evaluation needs **diverse, representative test cases** with expected qualities.

In [ ]:
TEST_CASES = [
    {'id': 'summarize_1', 'task': 'summarization',
     'input': 'The global semiconductor shortage that began in 2020 was caused by pandemic-driven demand shifts, factory shutdowns, and supply chain disruptions. Auto manufacturers were hit hardest with production cuts of up to 40%.',
     'criteria': 'concise, captures key facts, no hallucination'},
    {'id': 'explain_1', 'task': 'explanation',
     'input': 'Explain what a vector database is to a non-technical person.',
     'criteria': 'uses analogy, avoids jargon, accurate, under 100 words'},
    {'id': 'extract_1', 'task': 'extraction',
     'input': 'Extract company, revenue, year: Acme Corp reported $4.2B revenue for FY2023.',
     'criteria': 'correct extraction, structured output'},
    {'id': 'classify_1', 'task': 'classification',
     'input': 'Classify: My order arrived damaged, screen cracked, need replacement.',
     'criteria': 'correct category, confidence, brief reasoning'},
]
print(f'Defined {len(TEST_CASES)} test cases')

## 6 · Define Prompt Variants

We test three strategies: minimal, detailed-role, and chain-of-thought.

In [ ]:
PROMPT_VARIANTS = {
    'A_minimal': ChatPromptTemplate.from_messages([('human', '{input}')]),
    'B_detailed': ChatPromptTemplate.from_messages([
        ('system', 'You are a precise assistant. Be concise. Follow instructions exactly.'),
        ('human', '{input}'),
    ]),
    'C_cot': ChatPromptTemplate.from_messages([
        ('system', 'For every request: 1) Identify task type 2) Think about quality 3) Respond. Be concise.'),
        ('human', '{input}'),
    ]),
}
print(f'Defined {len(PROMPT_VARIANTS)} variants: {list(PROMPT_VARIANTS.keys())}')

## 7 · LLM-as-Judge Function

In [ ]:
def judge_response(test_case, response):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Score the response 1-5. Return ONLY JSON: {"relevance": N, "accuracy": N, "conciseness": N, "overall": N, "feedback": "..."}'),
        ('human', 'Input: {input}\nCriteria: {criteria}\nResponse: {response}'),
    ])
    chain = prompt | judge_llm | StrOutputParser()
    raw = chain.invoke({'input': test_case['input'], 'criteria': test_case['criteria'], 'response': response})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        if s >= 0 and e > s: return json.loads(raw[s:e])
    except json.JSONDecodeError: pass
    return {'overall': 3, 'feedback': 'Parse error'}

print('Judge function ready.')

## 8 · Run All Evaluations

In [ ]:
results = []
for tc in TEST_CASES:
    for vname, pt in PROMPT_VARIANTS.items():
        print(f'  {tc["id"]} x {vname}...', end=' ')
        chain = pt | llm | StrOutputParser()
        t0 = time.time()
        response = chain.invoke({'input': tc['input']})
        latency = time.time() - t0
        scores = judge_response(tc, response)
        results.append({
            'test_id': tc['id'], 'task': tc['task'], 'variant': vname,
            'response': response[:300], 'word_count': len(response.split()),
            'latency_s': round(latency, 2),
            **{k: v for k, v in scores.items()},
        })
        print(f'done ({latency:.1f}s, overall={scores.get("overall", "?")})')
print(f'Completed {len(results)} evaluations')

## 9 · Analyze Results

In [ ]:
df = pd.DataFrame(results)
print('RESULTS BY PROMPT VARIANT')
print('=' * 60)
num_cols = [c for c in ['overall','relevance','accuracy','conciseness','latency_s','word_count'] if c in df.columns]
print(df.groupby('variant')[num_cols].mean().round(2).to_string())

print('\nBEST VARIANT PER TEST CASE')
if 'overall' in df.columns:
    for tc_id in df['test_id'].unique():
        sub = df[df['test_id'] == tc_id]
        best = sub.loc[sub['overall'].idxmax()]
        print(f'  {tc_id}: {best["variant"]} (overall={best["overall"]})')

## 10 · Detailed Response Comparison

In [ ]:
tc_id = 'explain_1'
print(f'COMPARISON FOR: {tc_id}')
print('=' * 60)
for _, row in df[df['test_id'] == tc_id].iterrows():
    print(f'--- {row["variant"]} (overall={row.get("overall","?")}, words={row["word_count"]}) ---')
    print(row['response'][:200])

## 11 · Save Results

In [ ]:
df.to_csv('prompt_eval_results.csv', index=False)
with open('prompt_eval_results.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Saved {len(results)} results')

## 12 · Limitations

- **LLM-as-judge bias** — same model family may favor its own style
- **Small test set** — 4 test cases is not statistically robust
- **No human baseline** — should compare against human-written gold answers
- **Temperature variation** — single runs do not capture output variance

## 13 · How to Improve

1. Add more test cases (20+ per task type)
2. Multiple runs per variant to measure consistency
3. Cross-model judging
4. Human evaluation alongside LLM-as-judge

## 14 · Mini Challenge

1. Add a fourth prompt variant with few-shot examples
2. Add a consistency metric by running each variant 3 times
3. Create a variant optimized for the extraction task

## 15 · Key Takeaways

- Prompt evaluation = systematic comparison with test cases + scoring criteria
- LLM-as-judge enables automated multi-dimensional scoring
- Different prompt strategies work best for different task types
- Local-first with Ollama means fast iteration without API costs